# Marketing Pipeline V2

Ce notebook reconstruit proprement le mini-projet autour du dataset Kaggle **Customer Personality Analysis**.
L'objectif est double :

- segmenter les clients avec une approche non supervisée ;
- prédire la réponse à une campagne marketing avec une approche supervisée ;
- comparer un modèle baseline et un modèle hybride enrichi par `ClusterID`.

L'idée directrice est de corriger le biais de la première version en renforçant le preprocessing dès le début, tout en gardant une méthodologie simple, lisible et défendable dans un cadre académique.


## 2. Imports

On importe uniquement les bibliothèques utiles au projet, puis on définit les chemins de travail et quelques constantes pour garantir une exécution reproductible.


In [ ]:
import os
import warnings
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings(
    "ignore",
    message="Could not find the number of physical cores*",
    category=UserWarning,
)

RANDOM_STATE = 42
CURRENT_YEAR = 2026

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        raw_candidate = candidate / "data" / "raw" / "marketing_campaign.csv"
        if raw_candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Impossible de localiser le projet. Vérifiez la présence de data/raw/marketing_campaign.csv."
    )

PROJECT_ROOT = resolve_project_root()
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "marketing_campaign.csv"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "marketing_campaign_clustered_v2.csv"
BASELINE_RESULTS_PATH = PROJECT_ROOT / "reports" / "baseline_results_v2.csv"
HYBRID_RESULTS_PATH = PROJECT_ROOT / "reports" / "hybrid_results_v2.csv"

PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
BASELINE_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Processed data output: {PROCESSED_DATA_PATH}")
print(f"Reports directory: {BASELINE_RESULTS_PATH.parent}")


## 3. Chargement des données brutes

On vérifie d'abord que le fichier source existe bien, puis on charge le CSV brut avec le séparateur tabulaire attendu (`sep="	"`).


In [ ]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"Fichier source introuvable : {RAW_DATA_PATH}")

df_raw = pd.read_csv(RAW_DATA_PATH, sep="	")
df_raw.columns = df_raw.columns.str.strip()
df = df_raw.copy()

print(f"Dataset chargé avec succès : {df.shape[0]} lignes x {df.shape[1]} colonnes.")
display(df.head())


## 4. Inspection initiale

Cette étape permet de comprendre la structure du jeu de données avant toute transformation : dimensions, types, aperçu des valeurs manquantes et statistiques simples.


In [ ]:
print("Shape initiale :", df.shape)

print()
print("Types de données :")
display(pd.DataFrame({"dtype": df.dtypes.astype(str)}))

missing_initial = (
    df.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_pct=lambda x: (100 * x["missing_count"] / len(df)).round(2))
    .sort_values("missing_count", ascending=False)
)

print()
print("Résumé des valeurs manquantes :")
display(missing_initial)

overview_cols = [
    "Year_Birth",
    "Income",
    "Kidhome",
    "Teenhome",
    "MntWines",
    "NumWebPurchases",
    "Response",
]
print()
print("Statistiques descriptives sur quelques variables clés :")
display(df[overview_cols].describe().transpose())


## 5. Nettoyage

Le nettoyage retenu reste volontairement simple et explicable :

- nettoyage des noms de colonnes ;
- conversion de `Income` et `Response` en numérique ;
- suppression des lignes où `Income` ou `Response` sont manquants.


In [ ]:
rows_before_cleaning = len(df)

df.columns = df.columns.str.strip()
df["Income"] = pd.to_numeric(df["Income"], errors="coerce")
df["Response"] = pd.to_numeric(df["Response"], errors="coerce")

for col in ["Education", "Marital_Status"]:
    df[col] = df[col].astype("string").str.strip()

df = df.dropna(subset=["Income", "Response"]).copy()
df["Response"] = df["Response"].astype(int)

duplicate_rows = int(df.duplicated().sum())

print(f"Lignes supprimées car Income ou Response manquaient : {rows_before_cleaning - len(df)}")
print(f"Lignes dupliquées exactes détectées : {duplicate_rows}")
print(f"Shape après nettoyage de base : {df.shape}")


## 6. Feature engineering

On crée cinq variables synthétiques utiles pour l'analyse et la modélisation :

- `Age = 2026 - Year_Birth`
- `Children = Kidhome + Teenhome`
- `TotalSpending` comme somme des montants dépensés
- `TotalPurchases` comme somme des achats web, catalogue et magasin
- `IsParent` comme indicateur binaire dérivé de `Children`

On ajoute aussi quelques vérifications simples pour fiabiliser la suite du pipeline.


In [ ]:
spending_cols = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds",
]
purchase_cols = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]

df["Age"] = CURRENT_YEAR - df["Year_Birth"]
df["Children"] = df["Kidhome"].fillna(0) + df["Teenhome"].fillna(0)
df["IsParent"] = (df["Children"] > 0).astype(int)
df["TotalSpending"] = df[spending_cols].sum(axis=1)
df["TotalPurchases"] = df[purchase_cols].sum(axis=1)

print("Nouvelles variables créées : ['Age', 'Children', 'IsParent', 'TotalSpending', 'TotalPurchases']")
display(df[["ID", "Income", "Response", "Age", "Children", "IsParent", "TotalSpending", "TotalPurchases"]].head())

invalid_age_mask = ~df["Age"].between(18, 100)
print()
print(f"Âges hors plage plausible [18, 100] : {invalid_age_mask.sum()}")
if invalid_age_mask.any():
    display(
        df.loc[invalid_age_mask, ["ID", "Year_Birth", "Age"]]
        .sort_values("Age", ascending=False)
    )
    df = df.loc[~invalid_age_mask].copy()
    print("Ces lignes ont été retirées car leurs dates de naissance semblent aberrantes.")

if not pd.api.types.is_numeric_dtype(df["Income"]):
    raise TypeError("Income doit être numérique après nettoyage.")
print("Income est bien numérique.")

response_values = sorted(df["Response"].dropna().unique().tolist())
print(f"Modalités observées pour Response : {response_values}")
if not set(response_values).issubset({0, 1}):
    raise ValueError("Response doit être binaire (0/1).")
print("Response est bien binaire.")


## 7. Analyse des valeurs manquantes

Après nettoyage et feature engineering, on vérifie qu'il ne reste pas de valeurs manquantes susceptibles de perturber le clustering ou les modèles supervisés.


In [ ]:
missing_after = (
    df.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_pct=lambda x: (100 * x["missing_count"] / len(df)).round(2))
    .sort_values("missing_count", ascending=False)
)

display(missing_after)

if int(missing_after["missing_count"].sum()) == 0:
    print("Aucune valeur manquante restante après nettoyage et feature engineering.")
else:
    print("Des valeurs manquantes subsistent : elles devront être traitées avant la modélisation.")


## 8. Analyse / traitement simple des outliers

On cible ici la variable `Income`, car des valeurs extrêmes peuvent fortement influencer le clustering.
La logique retenue est volontairement simple : visualiser la distribution, puis conserver les revenus compris entre le 1er et le 99e percentile.


In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x=df["Income"], color="#7AA6C2")
plt.title("Income avant traitement des outliers")
plt.xlabel("Income")
plt.show()


In [ ]:
income_p01 = df["Income"].quantile(0.01)
income_p99 = df["Income"].quantile(0.99)

rows_before_outlier_filter = len(df)
df = df[df["Income"].between(income_p01, income_p99)].copy()
rows_removed_outliers = rows_before_outlier_filter - len(df)

print(f"Seuil bas (1er percentile) : {income_p01:,.2f}")
print(f"Seuil haut (99e percentile) : {income_p99:,.2f}")
print(f"Lignes retirées lors du filtrage des outliers Income : {rows_removed_outliers}")
print(
    "Logique retenue : limiter l'effet des revenus extrêmes tout en gardant un preprocessing simple,"
    " robuste et facile à justifier."
)

plt.figure(figsize=(10, 4))
sns.boxplot(x=df["Income"], color="#E59866")
plt.title("Income après filtrage entre le 1er et le 99e percentile")
plt.xlabel("Income")
plt.show()


## 9. Préparation des données pour le clustering

Le clustering repose sur un mélange de variables numériques et catégorielles.
On prépare donc les données avec un `ColumnTransformer` :

- `StandardScaler` pour les variables numériques ;
- `OneHotEncoder(handle_unknown="ignore")` pour les variables catégorielles.


In [ ]:
cluster_numeric_features = ["Income", "Age", "Children", "TotalSpending", "TotalPurchases"]
cluster_categorical_features = ["Education", "Marital_Status"]
cluster_features = cluster_numeric_features + cluster_categorical_features

if int(df[cluster_features].isna().sum().sum()) != 0:
    raise ValueError("Des valeurs manquantes subsistent dans les variables du clustering.")

cluster_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), cluster_numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cluster_categorical_features),
    ]
)

X_cluster = cluster_preprocessor.fit_transform(df[cluster_features])
X_cluster_prepared = X_cluster.toarray() if hasattr(X_cluster, "toarray") else X_cluster

print("Variables numériques du clustering :", cluster_numeric_features)
print("Variables catégorielles du clustering :", cluster_categorical_features)
print("Shape après transformation :", X_cluster_prepared.shape)


## 10. Recherche du meilleur nombre de clusters

On teste plusieurs valeurs de `k` entre 2 et 7 et on retient celle qui maximise le **silhouette score**.
Ce critère reste simple à expliquer : plus il est élevé, meilleure est la séparation moyenne entre les groupes.


In [ ]:
silhouette_records = []

for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    cluster_labels = kmeans.fit_predict(X_cluster_prepared)
    score = silhouette_score(X_cluster_prepared, cluster_labels)
    silhouette_records.append({"k": k, "silhouette_score": score})

silhouette_df = pd.DataFrame(silhouette_records)
best_k = int(silhouette_df.loc[silhouette_df["silhouette_score"].idxmax(), "k"])

display(silhouette_df.round(4))
print(f"Meilleur nombre de clusters retenu automatiquement : k = {best_k}")

plt.figure(figsize=(8, 4))
sns.lineplot(data=silhouette_df, x="k", y="silhouette_score", marker="o")
plt.title("Silhouette score en fonction de k")
plt.xlabel("Nombre de clusters (k)")
plt.ylabel("Silhouette score")
plt.xticks(range(2, 8))
plt.show()


## 11. Entraînement K-Means final

Une fois `k` choisi, on réentraîne le modèle K-Means final et on ajoute la variable `ClusterID` au DataFrame principal.


In [ ]:
kmeans_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)

df_clustered = df.copy()
df_clustered["ClusterID"] = kmeans_final.fit_predict(X_cluster_prepared)

print(f"K-Means final entraîné avec k = {best_k}")
print("Répartition des observations par cluster :")
print(df_clustered["ClusterID"].value_counts().sort_index())

display(df_clustered[["ID", "Income", "Age", "TotalSpending", "TotalPurchases", "ClusterID"]].head())


## 12. Analyse métier des clusters

On examine maintenant la structure des segments créés pour vérifier qu'ils sont lisibles d'un point de vue business.
Les visualisations suivantes permettent d'observer la taille des clusters, leur niveau de dépense et leur lien avec la variable `Response`.


In [ ]:
cluster_order = sorted(df_clustered["ClusterID"].unique())

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.countplot(data=df_clustered, x="ClusterID", order=cluster_order, color="#4C78A8", ax=axes[0])
axes[0].set_title("Répartition des clients par cluster")
axes[0].set_xlabel("ClusterID")
axes[0].set_ylabel("Nombre de clients")

sns.scatterplot(
    data=df_clustered,
    x="Income",
    y="TotalSpending",
    hue="ClusterID",
    palette="tab10",
    alpha=0.75,
    s=70,
    ax=axes[1],
)
axes[1].set_title("Income vs TotalSpending par cluster")
axes[1].set_xlabel("Income")
axes[1].set_ylabel("TotalSpending")

plt.tight_layout()
plt.show()


In [ ]:
cluster_response = df_clustered.groupby("ClusterID", as_index=False)["Response"].mean()
cluster_response["ResponseRatePct"] = 100 * cluster_response["Response"]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(
    data=cluster_response,
    x="ClusterID",
    y="ResponseRatePct",
    order=cluster_order,
    color="#59A14F",
    ax=axes[0],
)
axes[0].set_title("Taux de réponse par cluster")
axes[0].set_xlabel("ClusterID")
axes[0].set_ylabel("Taux de réponse (%)")

sns.boxplot(
    data=df_clustered,
    x="ClusterID",
    y="TotalSpending",
    order=cluster_order,
    color="#E15759",
    ax=axes[1],
)
axes[1].set_title("TotalSpending par cluster")
axes[1].set_xlabel("ClusterID")
axes[1].set_ylabel("TotalSpending")

plt.tight_layout()
plt.show()

pca_projection = PCA(n_components=2).fit_transform(X_cluster_prepared)
pca_df = pd.DataFrame(pca_projection, columns=["PCA1", "PCA2"])
pca_df["ClusterID"] = df_clustered["ClusterID"].values

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x="PCA1", y="PCA2", hue="ClusterID", palette="tab10", alpha=0.75, s=70)
plt.title("Projection PCA des clusters")
plt.xlabel("Composante principale 1")
plt.ylabel("Composante principale 2")
plt.show()


In [ ]:
cluster_summary = (
    df_clustered.groupby("ClusterID")
    .agg(
        Count=("ClusterID", "size"),
        Income_mean=("Income", "mean"),
        Age_mean=("Age", "mean"),
        Children_mean=("Children", "mean"),
        IsParent_rate=("IsParent", "mean"),
        TotalSpending_mean=("TotalSpending", "mean"),
        TotalPurchases_mean=("TotalPurchases", "mean"),
        ResponseRate=("Response", "mean"),
    )
    .round(2)
)

spend_q1, spend_q3 = cluster_summary["TotalSpending_mean"].quantile([0.25, 0.75])
income_q1, income_q3 = cluster_summary["Income_mean"].quantile([0.25, 0.75])
response_q1, response_q3 = cluster_summary["ResponseRate"].quantile([0.25, 0.75])

def interpret_cluster(row: pd.Series) -> str:
    if row["TotalSpending_mean"] >= spend_q3:
        spending_desc = "forte dépense"
    elif row["TotalSpending_mean"] <= spend_q1:
        spending_desc = "dépense plus modeste"
    else:
        spending_desc = "dépense intermédiaire"

    if row["Income_mean"] >= income_q3:
        income_desc = "revenu élevé"
    elif row["Income_mean"] <= income_q1:
        income_desc = "revenu plus limité"
    else:
        income_desc = "revenu intermédiaire"

    if row["ResponseRate"] >= response_q3:
        response_desc = "forte probabilité de réponse"
    elif row["ResponseRate"] <= response_q1:
        response_desc = "faible probabilité de réponse"
    else:
        response_desc = "réactivité moyenne"

    return f"{spending_desc}, {income_desc}, {response_desc}"

cluster_summary["ResponseRate_pct"] = (100 * cluster_summary["ResponseRate"]).round(2)
cluster_summary["Interpretation"] = cluster_summary.apply(interpret_cluster, axis=1)

display(
    cluster_summary[
        [
            "Count",
            "Income_mean",
            "Age_mean",
            "Children_mean",
            "IsParent_rate",
            "TotalSpending_mean",
            "TotalPurchases_mean",
            "ResponseRate_pct",
            "Interpretation",
        ]
    ]
)

highest_spending_cluster = int(cluster_summary["TotalSpending_mean"].idxmax())
highest_response_cluster = int(cluster_summary["ResponseRate"].idxmax())
lowest_income_cluster = int(cluster_summary["Income_mean"].idxmin())

print(f"Le cluster {highest_spending_cluster} correspond au segment à la plus forte dépense moyenne.")
print(f"Le cluster {highest_response_cluster} présente le meilleur taux de réponse observé.")
print(f"Le cluster {lowest_income_cluster} représente le segment au revenu moyen le plus faible.")


## 13. Sauvegarde du dataset enrichi avec ClusterID

On exporte maintenant la version enrichie du dataset, qui servira de trace de travail et pourra être réutilisée dans d'autres scripts ou notebooks.


In [ ]:
df_clustered.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"Dataset enrichi sauvegardé : {PROCESSED_DATA_PATH}")
print(f"Shape sauvegardée : {df_clustered.shape}")


## 14. Baseline model (sans ClusterID)

Pour la comparaison supervisée, on utilise volontairement un `RandomForestClassifier` simple, rapide et stable.
On garde le même split train/test pour la baseline et l'hybride afin de rendre la comparaison plus juste.


In [ ]:
baseline_features = ["Income", "Age", "Children", "TotalSpending", "TotalPurchases"]
hybrid_features = baseline_features + ["ClusterID"]
target_col = "Response"

model_df = df_clustered[["ID"] + hybrid_features + [target_col]].copy()

if int(model_df.isna().sum().sum()) != 0:
    raise ValueError("Des valeurs manquantes subsistent dans les variables de modélisation.")

train_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.2,
    stratify=model_df[target_col],
    random_state=RANDOM_STATE,
)

train_df = model_df.loc[train_idx].copy()
test_df = model_df.loc[test_idx].copy()
y_train = train_df[target_col]
y_test = test_df[target_col]

print(f"Split partagé pour la comparaison : {len(train_df)} lignes train / {len(test_df)} lignes test.")
print(f"Taux de réponse train : {y_train.mean():.3f}")
print(f"Taux de réponse test : {y_test.mean():.3f}")

rf_params = {
    "n_estimators": 300,
    "max_depth": 6,
    "min_samples_leaf": 2,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
}

print("Paramètres du RandomForest :", rf_params)

def evaluate_model(model_name: str, X_train: pd.DataFrame, X_test: pd.DataFrame):
    model = RandomForestClassifier(**rf_params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print()
    print(f"Classification report - {model_name}")
    print(classification_report(y_test, y_pred, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion matrix - {model_name}")
    plt.xlabel("Prédit")
    plt.ylabel("Réel")
    plt.show()

    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    }

    results_df = pd.DataFrame(
        {
            "ID": test_df["ID"].values,
            "y_true": y_test.values,
            "y_pred": y_pred,
            "y_prob": y_prob,
        }
    )

    return model, results_df, metrics, cm


In [ ]:
X_train_baseline = train_df[baseline_features]
X_test_baseline = test_df[baseline_features]

baseline_model, baseline_results_df, baseline_metrics, baseline_cm = evaluate_model(
    model_name="Baseline (sans ClusterID)",
    X_train=X_train_baseline,
    X_test=X_test_baseline,
)

print("Métriques baseline :")
display(pd.DataFrame([baseline_metrics]).round(4))


## 15. Hybrid model (avec ClusterID)

On reprend exactement la même logique, mais on ajoute maintenant `ClusterID` aux variables explicatives pour tester l'hypothèse du projet.


In [ ]:
X_train_hybrid = train_df[hybrid_features]
X_test_hybrid = test_df[hybrid_features]

hybrid_model, hybrid_results_df, hybrid_metrics, hybrid_cm = evaluate_model(
    model_name="Hybrid (avec ClusterID)",
    X_train=X_train_hybrid,
    X_test=X_test_hybrid,
)

print("Métriques hybrid :")
display(pd.DataFrame([hybrid_metrics]).round(4))


## 16. Comparaison des performances

On compare ici les deux approches à l'aide de cinq métriques simples : accuracy, precision, recall, F1-score et ROC-AUC.
L'interprétation doit rester prudente : un gain peut dépendre du split, du choix de `k` et du modèle supervisé utilisé.


In [ ]:
comparison_df = pd.DataFrame(
    [baseline_metrics, hybrid_metrics],
    index=["Baseline", "Hybrid"],
).round(4)

display(comparison_df)

improvement_df = (comparison_df.loc["Hybrid"] - comparison_df.loc["Baseline"]).to_frame("Hybrid - Baseline")
display(improvement_df.round(4))

positive_metrics = improvement_df[improvement_df["Hybrid - Baseline"] > 0].index.tolist()
if positive_metrics:
    print(f"Le modèle hybride améliore les métriques suivantes : {', '.join(positive_metrics)}.")
else:
    print("Le modèle hybride n'améliore aucune métrique dans cette configuration.")

if comparison_df.loc["Hybrid", "ROC-AUC"] > comparison_df.loc["Baseline", "ROC-AUC"]:
    print("Dans ce rerun, l'ajout de ClusterID améliore légèrement la capacité de discrimination globale (ROC-AUC).")
else:
    print("Dans ce rerun, l'ajout de ClusterID n'améliore pas la ROC-AUC, même s'il peut aider d'autres métriques.")

print(
    "L'amélioration éventuelle doit être interprétée avec prudence : l'intérêt du projet est aussi méthodologique et analytique,"
    " car la segmentation apporte une lecture business complémentaire des profils clients."
)

stability_seeds = [7, 21, 42, 84, 123]

def evaluate_stability(feature_cols):
    records = []
    for seed in stability_seeds:
        train_idx_seed, test_idx_seed = train_test_split(
            model_df.index,
            test_size=0.2,
            stratify=model_df[target_col],
            random_state=seed,
        )
        y_train_seed = model_df.loc[train_idx_seed, target_col]
        y_test_seed = model_df.loc[test_idx_seed, target_col]
        model = RandomForestClassifier(**{**rf_params, "random_state": seed})
        model.fit(model_df.loc[train_idx_seed, feature_cols], y_train_seed)
        y_pred_seed = model.predict(model_df.loc[test_idx_seed, feature_cols])
        y_prob_seed = model.predict_proba(model_df.loc[test_idx_seed, feature_cols])[:, 1]
        records.append(
            {
                "Seed": seed,
                "Accuracy": accuracy_score(y_test_seed, y_pred_seed),
                "Precision": precision_score(y_test_seed, y_pred_seed, zero_division=0),
                "Recall": recall_score(y_test_seed, y_pred_seed, zero_division=0),
                "F1-score": f1_score(y_test_seed, y_pred_seed, zero_division=0),
                "ROC-AUC": roc_auc_score(y_test_seed, y_prob_seed),
            }
        )
    return pd.DataFrame(records)

baseline_stability_df = evaluate_stability(baseline_features)
baseline_stability_df["Model"] = "Baseline"
hybrid_stability_df = evaluate_stability(hybrid_features)
hybrid_stability_df["Model"] = "Hybrid"
stability_df = pd.concat([baseline_stability_df, hybrid_stability_df], ignore_index=True)

stability_summary_df = (
    stability_df.groupby("Model")[["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]]
    .agg(["mean", "std"])
    .round(4)
)
display(stability_summary_df)
print("Ce tableau donne une lecture simple de la stabilité relative des deux approches sur plusieurs seeds.")


## 17. Export des résultats

On exporte enfin les prédictions test des deux modèles au format CSV, avec la vérité terrain, la classe prédite et la probabilité estimée.


In [ ]:
baseline_results_df.to_csv(BASELINE_RESULTS_PATH, index=False)
hybrid_results_df.to_csv(HYBRID_RESULTS_PATH, index=False)

print(f"Résultats baseline exportés : {BASELINE_RESULTS_PATH}")
print(f"Résultats hybrid exportés : {HYBRID_RESULTS_PATH}")
print(f"Nombre de lignes exportées (baseline) : {len(baseline_results_df)}")
print(f"Nombre de lignes exportées (hybrid) : {len(hybrid_results_df)}")


## 18. Conclusion

Cette seconde version propose un pipeline plus propre et plus défendable : nettoyage explicite, contrôles simples, traitement mesuré des outliers et comparaison rigoureuse entre baseline et modèle hybride.


In [ ]:
best_model_label = comparison_df["ROC-AUC"].idxmax()

print(
    "Le preprocessing renforcé a permis de repartir d'une base plus fiable : suppression des valeurs critiques manquantes,"
    " retrait de quelques âges aberrants et filtrage simple des revenus extrêmes."
)
print(f"Le clustering K-Means retient ici k = {best_k} selon le silhouette score.")
print(f"Le meilleur modèle sur la ROC-AUC observée dans ce notebook est : {best_model_label}.")
print(
    "Au-delà du score pur, l'intérêt du notebook est aussi de montrer comment la segmentation peut enrichir"
    " la lecture marketing du dataset et nourrir une analyse plus structurée."
)


## Limites et pistes d'amélioration

Quelques prolongements naturels sont possibles pour aller plus loin :

- tester d'autres algorithmes de clustering, par exemple `AgglomerativeClustering` ou `DBSCAN` ;
- réaliser un tuning plus systématique des hyperparamètres du modèle supervisé ;
- ajouter une validation croisée pour stabiliser l'évaluation ;
- transformer ce notebook en pipeline plus industrialisable avec scripts dédiés, journalisation et suivi d'expériences.
